In [1]:
import os
import json
import time
import requests
from dotenv import load_dotenv

# ✅ Load API key from .env (or Docker environment)
load_dotenv()
API_KEY = os.getenv("GOOGLE_MAP_API_KEY")

if not API_KEY:
    raise ValueError("❌ GOOGLE_MAP_API_KEY not found in environment.")

# --- CONFIG ---
SOURCE_URL = "https://raw.githubusercontent.com/ThathsaraniPathirana/LLM-project/refs/heads/main/Places/all_places_sweden_flat.json"
OUTPUT_FILE = "ratings_places.json"
BATCH_SIZE = 50
SLEEP_SEC = 1

# --- Load source data ---
print("📂 Downloading source JSON...")
response = requests.get(SOURCE_URL)
response.raise_for_status()
data = response.json()
print(f"✅ Loaded {len(data)} records from source file.")

# --- Load existing progress if available ---
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        ratings = json.load(f)
    processed_names = {r["name"] for r in ratings}
    print(f"🔁 Resuming — already processed {len(processed_names)} places.")
else:
    ratings = []
    processed_names = set()

# --- Define Google Places endpoint ---
url = "https://places.googleapis.com/v1/places:searchText"
headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.displayName,places.rating,places.userRatingCount,places.formattedAddress,places.googleMapsUri"
}

# --- Main loop ---
count = 0
for record in data:
    name = record.get("name")
    alt = record.get("alternate_name")
    street = record.get("street")
    city = record.get("city")

    if not name or name in processed_names:
        continue  # Skip invalid or already processed

    # Build text query using available fields
    parts = [name, alt, street, city, "Sweden"]
    query = ", ".join([p for p in parts if p])
    payload = {"textQuery": query}

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=15)
        result = response.json()

        if "error" in result:
            print(f"⚠️ Error for {name}: {result['error'].get('message')}")
            continue

        places = result.get("places", [])
        if not places:
            print(f"❌ No result found for: {name}")
            continue

        place = places[0]
        entry = {
            "name": name,
            "alternate_name": alt,
            "rating": place.get("rating"),
            "userRatingCount": place.get("userRatingCount"),
            "formattedAddress": place.get("formattedAddress"),
            "googleMapsUri": place.get("googleMapsUri"),
        }

        ratings.append(entry)
        processed_names.add(name)
        count += 1

        print(f"✅ [{count}] {name} → ⭐ {entry['rating']} ({entry['userRatingCount']})")

        # Periodic save
        if count % BATCH_SIZE == 0:
            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(ratings, f, ensure_ascii=False, indent=2)
            print(f"💾 Progress saved — {count} places done.")
            time.sleep(2)  # small pause between batches

        time.sleep(SLEEP_SEC)  # Respect rate limits

    except Exception as e:
        print(f"⚠️ Error for {name}: {e}")
        time.sleep(3)
        continue

# --- Final save ---
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(ratings, f, ensure_ascii=False, indent=2)

print(f"\n🎯 Completed! Saved {len(ratings)} results to {OUTPUT_FILE}")


📂 Downloading source JSON...
✅ Loaded 7231 records from source file.
✅ [1] A little party → ⭐ 5 (10)
❌ No result found for: ALPINPAKET PÅ NYBOHOLM
✅ [2] Store Mosse nationalpark → ⭐ 4.6 (2046)
✅ [3] Glaskogens Naturreservat → ⭐ 4.7 (812)
✅ [4] Krigsflygfält 16 → ⭐ 4.6 (89)
✅ [5] Tre toppar → ⭐ 4.4 (8)
✅ [6] Team Sportia Säffle → ⭐ 4.4 (75)
✅ [7] Alkvetterns fiskeförening → ⭐ 4.7 (10)
✅ [8] Fiske i Glaskogens Naturreservat → ⭐ 4.7 (812)
✅ [9] Nolbygårds Ekobageri → ⭐ 4.6 (539)
✅ [10] Brödfabriken i Jonsered → ⭐ 4.7 (485)
✅ [11] V2 Alleri → ⭐ 5 (2)
✅ [12] Poesiplatsen → ⭐ 4.5 (716)
✅ [13] VKM → ⭐ 4.2 (2233)
✅ [14] Kusthotellet Styrsö → ⭐ 4.7 (14)
✅ [15] Bistro Tuppen → ⭐ 4.2 (156)
✅ [16] Stora Oset → ⭐ 4.4 (127)
✅ [17] ÖHK-hallen → ⭐ 4 (158)
✅ [18] Bed & Bike Öckerö → ⭐ 4.6 (18)
✅ [19] Fred’s Food and Coffee → ⭐ 4.2 (48)
✅ [20] Hinsholmen Mat & Event → ⭐ 4.5 (37)
✅ [21] Flygarns Haga → ⭐ 4.4 (53)
✅ [22] Skaterampen i Frihamnen → ⭐ 4.1 (409)
✅ [23] Bar à Kaffe → ⭐ 4.7 (37)
✅ [24] Kaiser →

TypeError: sequence item 1: expected str instance, dict found

### Code resuming

In [4]:
import os
import json
import time
import requests
from dotenv import load_dotenv

# -------------------------------------------
# Load API key
# -------------------------------------------
load_dotenv()
API_KEY = os.getenv("GOOGLE_MAP_API_KEY")

if not API_KEY:
    raise ValueError("❌ GOOGLE_MAP_API_KEY not found in environment.")

# -------------------------------------------
# Config
# -------------------------------------------
SOURCE_URL = "https://raw.githubusercontent.com/ThathsaraniPathirana/LLM-project/refs/heads/main/Places/all_places_sweden_flat.json"
OUTPUT_FILE = "ratings_places.json"
BATCH_SIZE = 50
SLEEP_SEC = 1

# -------------------------------------------
# Load source data
# -------------------------------------------
print("📂 Downloading source JSON...")
response = requests.get(SOURCE_URL)
response.raise_for_status()
data = response.json()
print(f"✅ Loaded {len(data)} records from source file.")

# -------------------------------------------
# Resume progress
# -------------------------------------------
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        ratings = json.load(f)
    processed_names = {r["name"] for r in ratings}
    print(f"🔁 Resuming — already processed {len(processed_names)} places.")
else:
    ratings = []
    processed_names = set()

url = "https://places.googleapis.com/v1/places:searchText"
headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.displayName,places.rating,places.userRatingCount,places.formattedAddress,places.googleMapsUri"
}

# -------------------------------------------
# Helper to extract plain string from dict
# -------------------------------------------
def extract_value(v):
    if isinstance(v, dict):
        return v.get("@value") or None
    return v

# -------------------------------------------
# Processing loop
# -------------------------------------------
count = len(ratings)
for record in data:
    name = record.get("name")
    alt = record.get("alternate_name")
    street = extract_value(record.get("street"))
    city = extract_value(record.get("city"))

    if not name or name in processed_names:
        continue  # Skip duplicates / already processed

    parts = [name, alt, street, city, "Sweden"]
    query = ", ".join([str(p) for p in parts if p])
    payload = {"textQuery": query}

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=15)
        result = response.json()

        if "error" in result:
            print(f"⚠️ Error for {name}: {result['error'].get('message')}")
            continue

        places = result.get("places", [])
        if not places:
            print(f"❌ No result found for: {name}")
            continue  # skip without saving

        place = places[0]
        entry = {
            "name": name,
            "alternate_name": alt,
            "rating": place.get("rating"),
            "userRatingCount": place.get("userRatingCount"),
            "formattedAddress": place.get("formattedAddress"),
            "googleMapsUri": place.get("googleMapsUri"),
        }

        ratings.append(entry)
        processed_names.add(name)
        count += 1

        print(f"✅ [{count}] {name} → ⭐ {entry['rating']} ({entry['userRatingCount']})")

        # Save progress every batch
        if count % BATCH_SIZE == 0:
            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(ratings, f, ensure_ascii=False, indent=2)
            print(f"💾 Progress saved — {count} places done.")
            time.sleep(2)

        time.sleep(SLEEP_SEC)

    except Exception as e:
        print(f"⚠️ Error for {name}: {e}")
        time.sleep(3)
        continue

# -------------------------------------------
# Final save
# -------------------------------------------
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(ratings, f, ensure_ascii=False, indent=2)

print(f"\n🎯 Completed! Saved {len(ratings)} results to {OUTPUT_FILE}")


📂 Downloading source JSON...
✅ Loaded 7231 records from source file.
🔁 Resuming — already processed 1150 places.
❌ No result found for: ALPINPAKET PÅ NYBOHOLM
❌ No result found for: Algblomman
❌ No result found for: Papi’s Pierogi
❌ No result found for: Elite Tours: Exclusive Experiences Sweden
❌ No result found for: Sophie Mess
❌ No result found for: Interhome
❌ No result found for: Ringlinien
✅ [1151] Beyond Retro → ⭐ 4.4 (844)
✅ [1152] Bhoga → ⭐ 4.7 (455)
✅ [1153] Draken Live → ⭐ 4.1 (935)
✅ [1154] Bee Kök & Bar → ⭐ 4.1 (1596)
✅ [1155] Bokskåpet → ⭐ 4.8 (106)
✅ [1156] Backa Teater → ⭐ 4.2 (196)
✅ [1157] Hôtel Eggers → ⭐ 4.3 (1848)
✅ [1158] Bike Tour Gothenburg & Rental → ⭐ 4.9 (124)
✅ [1159] Bagaren och Kocken → ⭐ 3.8 (390)
✅ [1160] Biopalatset → ⭐ 4.2 (3900)
✅ [1161] Bar Centro → ⭐ 4.6 (713)
✅ [1162] Båtebackens Caférestaurang → ⭐ 4.1 (211)
✅ [1163] Bengans Skivbutik → ⭐ 4.6 (1441)
✅ [1164] Biljardpalatset & BP Brasserie → ⭐ 4 (2272)
✅ [1165] Bergsjöbadet → ⭐ None (None)
✅ [1166] B